# Synthetic Data Generator - Week 3 Assignment

## Submitted By: Isaac Ifeakachi Nwankwo (issoft2)

# Summary

- Implemented a synthetic data generator using the transformer architure directly.
- Used AutoTokenizer and AutoModelForCausalLM for manual inference.
- Demonstrated core transformer flow:Tokenize ---> Generate ---> Decode.
- Wrapped the logic in a Gradio UI for usability.
- Used a small model
- Fully aligned with week 3 challenge: "Write models that generate datasets and explore model APIs."

# Pip installations

In [ ]:
!pip install -q transformers gradio torch

In [ ]:
# @title Default title text
# Let's chec the GPU - It should be NVIDIA Tesla T4
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('Tesla T4') >= 0:
    print('GPU is Tesla T4')
else:   
    print('GPU is not Tesla T4')       

In [ ]:
# Import libraries
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import gradio as gr
from huggingface_hub import login
from google.colab import userdata


In [ ]:
# Login to Hugging Face Hub
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
# Load Model and Tokenizer
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)


# Build a prompt

Create a simple function to structure the generation task

In [ ]:
def build_prompt(region, count):
    return (
        f"Generate {count} unique Nigeria names from the {region} region. "
        f"Include both male and femaile names. "
        f"Return the list numbered 1 to {count}."
    )

# Tokenize, Generate and Decode

In [ ]:
def gnerate_names(region, count):
    # Few examples to guide GTP2
    prompt = f"""
Generate {count} unique Nigerian names from the {region} region.and
Each name should be real and commonly used in that region in Nigeria
Include both male and female names.
Here are some examples:

1. Jude Okafor
2. Adeola Adebayo
3. Chinedu Eze
4. Fatima Bello
5. Emeka Nwosu

Now go ahead and generate the list of names based on the above examples.:
"""

    # Model and tokenizer ...
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    # Encode the input --
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate output ...
    outputs = model.generate(
        **inputs,
        max_length=300,
        temperature=0.7,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the output ...
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract the names from the generated text ...
    lines = generated_text.split('\n')
    names = []
    for line in lines:
        if any(ch.isalpha() for ch in line):
            name = line.strip()
            if "," in name:
                name = name.split(",", 1)[1].strip()
            if len(name.split()) >= 3 and not name.lower().startswith("generate"):
                names.append(name)

    names = list(dict.fromkeys(names))[:count]

    if not names:
        names = ["No names generated, please try again."]

    return "\n".join(names)     

In [ ]:
# Catch error
import traceback

def safe_generate_names(region, count):
    try:
        return generate_names(region, count)
    except Exception as e:
        return f"Error:\n{traceback.format_exc()}"

In [ ]:
# Gradio Interface

def run_application():
    with gr.Blocks() as demo:
        gr.Markdown("## Nigerian Name Generator")
        gr.Markdown("Enter a Nigerian region and the number of names you want to generate. The model will return a list of unique Nigerian names from that region.")

        region = gr.Dropdown(
            ["Yoruba", "Igbo", "Hausa", "Fulani", "Kanuri", "Tiv", "Ibibio", "Edo", "Nupe", "Ijaw"],
            label="Select Region"
            value="Yoruba"

        )

        count = gr.Number(label="Number of Names", value=20)
        output = gr.Textbox(label="Generated Nigeria Names", lines=20)
        generate_button = gr.Button("Generate Names")

        generate_button.click(fn=gnerate_names, inputs=[region, count], outputs=output)

    demo.launch()


# Run App
    

In [ ]:
run_application()